# Rock–Paper–Scissors Mind Reader ✊✋✌️
Humans are terrible random number generators: we avoid repeating, we follow patterns, we copy what just won. A bot that **predicts your next move from your last k moves** (a Markov / n-gram model, like a tiny language model over 3 "words") beats most people. Build it, test it on patterned bots, then play it yourself.

Background: [[Probability for ML]], [[NLP Overview]] (n-gram models) · overview: [[Fun Projects]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
MOVES = ["rock", "paper", "scissors"]          # 0, 1, 2; move (m+1) % 3 beats m

## 1. Predict the next move
Find every earlier occurrence of the opponent's last `k` moves in their history, count what came **next**, and return the most frequent follow-up (random if there's no data yet).

In [ ]:
def predict_next(history, k, rng):
    # TODO 1: n-gram prediction from the opponent's own history
    raise NotImplementedError  # TODO 1

def counter(move):
    # TODO 2: the move that beats `move`
    raise NotImplementedError  # TODO 2

In [ ]:
def _t_pred():
    rng = np.random.default_rng(0)
    return predict_next([0, 1, 2, 0, 1, 2, 0, 1], 2, rng) == 2 and predict_next([0, 0, 0, 0], 1, rng) == 0 and [counter(m) for m in range(3)] == [1, 2, 0]
check("predictor", _t_pred, "After …0,1 the history says 2 comes next; paper beats rock, scissors beats paper, rock beats scissors.")

def cycler(my_hist, opp_hist, rng):        # rock → paper → scissors → …, with 10 % noise
    return int(rng.integers(3)) if rng.random() < 0.1 or not my_hist else (my_hist[-1] + 1) % 3
def copycat(my_hist, opp_hist, rng):       # plays what the opponent played last time
    return int(rng.integers(3)) if not opp_hist else opp_hist[-1]
def randomer(my_hist, opp_hist, rng): return int(rng.integers(3))

def match(opponent, k=2, rounds=500, seed=0):
    rng = np.random.default_rng(seed); bot_h, opp_h, score = [], [], []
    for _ in range(rounds):
        mine = counter(predict_next(opp_h, k, rng)); theirs = opponent(opp_h, bot_h, rng)
        score.append(0 if mine == theirs else 1 if mine == (theirs + 1) % 3 else -1)
        bot_h.append(mine); opp_h.append(theirs)
    s = np.array(score); return (s == 1).mean(), (s == -1).mean()
if ready("predictor"):
    res = {name: match(fn) for name, fn in [("cycler", cycler), ("copycat", copycat), ("random", randomer)]}
    for n, (w, l) in res.items(): print(f"vs {n:<8} win {w:.0%}  lose {l:.0%}")
    check("reads minds", lambda: res["cycler"][0] > 0.7 and abs(res["random"][0] - res["random"][1]) < 0.1,
          "Should win > 70 % against the cycler and roughly break even against true randomness.")
    print("→ The copycat only reacts to the BOT's moves, which our predictor ignores, so it isn't beaten. Second stretch goal: add the bot's own history as context.")

## 2. Play it yourself
Run `play_me(30)` and type r/p/s. Can you stay unpredictable for 30 rounds?

In [ ]:
def play_me(rounds=30, k=2):
    rng = np.random.default_rng(); h, wins = [], [0, 0]
    for _ in range(rounds):
        bot = counter(predict_next(h, k, rng)); you = "rps".index(input("r/p/s: ").strip()[0])
        if bot == (you + 1) % 3: wins[1] += 1
        elif you == (bot + 1) % 3: wins[0] += 1
        h.append(you); print(f"bot played {MOVES[bot]} – you {wins[0]} : {wins[1]} bot")
# play_me()

<details><summary>▶ Solution</summary>

```python
def predict_next(history, k, rng):
    if len(history) < k + 1:
        return int(rng.integers(3))
    ctx = tuple(history[-k:])
    counts = np.zeros(3)
    for i in range(len(history) - k):
        if tuple(history[i:i + k]) == ctx:
            counts[history[i + k]] += 1
    return int(rng.integers(3)) if counts.sum() == 0 else int(np.argmax(counts))
```

```python
def counter(move):
    return (move + 1) % 3
```
</details>

## Stretch goals
- Ensemble several `k` (1, 2, 3, 4) and pick whichever predictor has been right most often recently: the idea behind the famous *Iocaine Powder* bot.
- Use both histories (yours **and** the bot's) as context: people react to what they just lost to.
- Measure the entropy of your own play ([[Information Theory]]): below log2 3 = 1.585 bits, you're predictable.